## Practice 3 Hugging Face Transformers: Sentiment Analysis and Binary Text Classification

**General Objective:**  
Become familiar with the Hugging Face Transformers ecosystem through two exercises:
- **Exercise 1:** Use an already fine-tuned model to perform inference.
- **Exercise 2:** Fine-tune a generic pretrained model for binary text classification.

### 1. Objective

- Distinguish between the two layers of knowledge: **Pretraining** (language knowledge) và **Downstream Fine-tuning**
  (sentiment classification knowledge).

| Exercise | Main Objective | Training Required? | Checkpoint Used |
|----------|----------------|--------------------|--------------------|
| **Exercise 1** | Inference + tokenizer understanding | Không | `distilbert-base-uncased-finetuned-sst-2-english` |
| **Exercise 2** | Full-process fine-tuning | Yes | `distilbert-base-uncased` (generic) |


### 2. Scientific and Theoretical Foundation

#### (a) Why Choose Rotten Tomatoes

| Criterion | Rotten Tomatoes | IMDb |
|---|---|---|
| Labeled samples | 10,662 | 50,000 |
| Train split | 8,530 | 25,000 |
| Validation split | 1,066 | No separate official split |
| Test split | 1,066 | 25,000 |
| Binary sentiment | Yes | Yes |
| Typical text length | Short | Longer |
| Fine-tuning cost | Lower | High hơn |
| Suitable for a compact practice notebook | Very high | High |
| Alignment with the official Hugging Face tutorial | Medium | Very high |

**Reason for choosing:** a binary classification task, a moderate computational workload for a practice notebook, three existing
train/validation/test splits, relatively short reviews, no need to create an additional validation split,
and support for a clean experimental workflow on a personal computer (CPU-only).

```
Train: 8,530
Validation: 1,066
Test: 1,066
Total: 10,662

Label 0: NEGATIVE
Label 1: POSITIVE
```

#### (b) Why Choose DistilBERT

```
flowchart LR
    A[Tokenized Text] --> B[DistilBERT Backbone]
    B --> C[Contextual Representation]
    C --> D[Classification Head]
    D --> E[Two Logits]
    E --> F[NEGATIVE or POSITIVE]
```

| Model | Main Advantages | Main Limitations |
|---|---|---|
| DistilBERT | Lightweight, practical | Lower capacity than BERT-base |
| BERT-base | Classic Transformer baseline | Higher computational cost |
| RoBERTa-base | Strong language representation | Higher computational cost |
| MiniLM | Very lightweight | Less aligned with the traditional introductory workflow |
| ALBERT | Parameter-efficient | Different architectural characteristics |

**Primary implementation choice: DistilBERT** - a pretrained Transformer that is lighter than BERT-base, directly supports
sequence classification, is suitable for English sentiment analysis, reduces training cost while preserving
the nature of the transfer-learning workflow, and is suitable for a CPU-only lab environment.

#### (c) Conceptual Nature of Transfer Learning

```
flowchart TD
    A[Pretraining] --> B[General Language Knowledge]
    B --> C[Downstream Fine-Tuning]
    C --> D[Binary Sentiment Knowledge]
    D --> E[Inference]
    E --> F[Positive or Negative Prediction]
```

```
Exercise 1
Already fine-tuned model
        |
        v
Inference

Exercise 2
Generic pretrained model
        |
        v
Task-specific fine-tuning
        |
        v
Binary classifier
```

#### (d) How Fine-tuning Differs from Feature Extraction

```
Feature extraction
Freeze Transformer backbone
        |
        v
Train only classifier

Fine-tuning
Update Transformer backbone
        +
Update classification head
```

| Criterion | Feature Extraction | Fine-tuning |
|----------|--------------------|-----------------------------------|
| Base model weights | Freeze (not updated) | Updated together with the classification head |
| Base model role | Only extracts fixed features | Adapts representations to fit the task |
| Computational cost | Lower | High hơn |
| Application in this practice | Not used | Yes (Trainer fine-tunes the entire model) |

Practice 3 (Exercise 2) applies **Fine-tuning** to the entire `distilbert-base-uncased` backbone, which is updated
together with the classification head; no layers are frozen.

#### (e) Things That Must Never Be Done Throughout Practice 3

```
Do not:
Fine-tune on the test set

Do not:
Use test performance to choose epochs or hyperparameters

Do not:
Aggressively remove stopwords, punctuation, or linguistic structure without justification

Do not:
Train DistilBERT from scratch for this practice

Do not:
Use an already sentiment-fine-tuned checkpoint in Exercise 2
and describe the process as generic pretrained-model fine-tuning
without explicitly documenting the checkpoint's prior task-specific training

Do not:
Fabricate loss curves, metrics, confusion matrices, or benchmark results
```

### 3. Overall End-to-End Pipeline Map

```
flowchart TD
    S[START] --> A[Environment and Seeds]

    A --> B[Exercise 1]
    B --> C[Load Fine-Tuned Sentiment Model]
    C --> D[Inspect Sentence Tokens]
    D --> E[Run Sentiment Inference]

    E --> F[Exercise 2]
    F --> G[Load Rotten Tomatoes Dataset]
    G --> H[EDA and Data Quality Checks]
    H --> I[Load DistilBERT Tokenizer]
    I --> J[Token-Length Analysis]
    J --> K[Tokenize Dataset]
    K --> L[Dynamic Padding]
    L --> M[Load Pretrained DistilBERT Classifier]
    M --> N[Model Sanity Check]
    N --> O[Define Metrics]
    O --> P[Configure TrainingArguments]
    P --> Q[Create Trainer]
    Q --> R[Debug Subset Run]
    R --> T{Pipeline Valid?}
    T -- No --> U[Fix and Re-run]
    U --> R
    T -- Yes --> V[Full Fine-Tuning]
    V --> W[Validation Monitoring]
    W --> X[Load Best Checkpoint]
    X --> Y[Final Test Evaluation]
    Y --> Z[Confusion Matrix]
    Z --> AA[Error Analysis]
    AA --> AB[New-Sentence Inference]
    AB --> AC[Save Model and Tokenizer]
    AC --> AD[Reload Sanity Test]
    AD --> AE[FINAL SUMMARY]
```

### 4. 16-Phase Diagram According to Notebook Architecture

```
flowchart TD
    P0[Phase 0<br/>Practice Overview] --> P1[Phase 1<br/>Environment and Reproducibility]
    P1 --> P2[Phase 2<br/>Pretrained Sentiment Inference]
    P2 --> P3[Phase 3<br/>Tokenization Investigation]
    P3 --> P4[Phase 4<br/>Dataset Loading]
    P4 --> P5[Phase 5<br/>EDA and Sanity Checks]
    P5 --> P6[Phase 6<br/>Tokenizer and Preprocessing]
    P6 --> P7[Phase 7<br/>Model Construction]
    P7 --> P8[Phase 8<br/>Metrics and Training Configuration]
    P8 --> P9[Phase 9<br/>Fine-Tuning]
    P9 --> P10[Phase 10<br/>Learning Curves]
    P10 --> P11[Phase 11<br/>Validation and Test Evaluation]
    P11 --> P12[Phase 12<br/>Error Analysis]
    P12 --> P13[Phase 13<br/>New-Sentence Inference]
    P13 --> P14[Phase 14<br/>Save and Reload]
    P14 --> P15[Phase 15<br/>Final Summary]
```

### 5. Stage 1 Scope

| Phase | Phase Name | Main Objective |
|-------|-----------|----------------|
| 0 | Practice Overview | Problem definition + academic architecture |
| 1 | Environment & Reproducibility | Set up environment, seed, device |
| 2 | Pretrained Sentiment Inference | Exercise 1 - Inference |
| 3 | Tokenization Investigation | Tokenizer analysis |
| 4 | Dataset Loading | Load Rotten Tomatoes |
| 5 | Dataset EDA & Sanity Checks | Data analysis |
| 6 | Tokenizer & Preprocessing | Prepare data for fine-tuning |


### 6. Key Technical Decisions

| Item | Decision | Status |
|----------|----------|--------|
| Model Exercise 1 | `distilbert-base-uncased-finetuned-sst-2-english` | Approved |
| Model Exercise 2 | `distilbert-base-uncased` | Approved |
| Dataset | Rotten Tomatoes | Approved |
| Hardware | CPU only | Approved |
| Random Seed | 42 | Approved |

### 7. Final Conceptual Summary

```
flowchart LR
    A[General Language Pretraining] --> B[Pretrained DistilBERT]
    B --> C[Binary Sentiment Fine-Tuning]
    C --> D[Task-Specific Classifier]
    D --> E[Validation]
    E --> F[Independent Test Evaluation]
    F --> G[Reusable Sentiment Model]
```

```
Exercise 1
Pretrained model reuse
        +
Tokenizer understanding

Exercise 2
Transfer learning
        +
Fine-tuning
        +
Scientific evaluation
        +
Reusable model artifacts
```

## Phase 1 Environment & Reproducibility

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root added: {project_root}") 

Project root added: /Users/vientu/Deep Learning/TOTAL_LABPARACTICE_FOR_DEEP_LEARNING/total_practice/practice_3


In [2]:
# Phase 1: Environment & Reproducibility
from processing_own_phase.phase_01_environment import (
    set_seed,
    print_environment_info,
    save_environment_report,
)

# Set seed (must be done before running anything else)
set_seed(42)

# Print full report and save JSON file
info = print_environment_info()
filepath = save_environment_report(info)

# Quick summary
print(f"\nReport saved: {filepath}")
print(f"Device: {info['device']}")
print(f"Seed: {info['seed']}")


ENVIRONMENT INFORMATION
Timestamp: 2026-08-12T19:13:51.574901
Python Version: 3.11.14
Device: mps
Global Seed: 42

--- Package Versions ---
[OK     ] transformers    installed: 5.14.1       required: 5.14.1
[OK     ] datasets        installed: 5.0.1        required: 5.0.1
[OK     ] evaluate        installed: 0.4.6        required: 0.4.6
[OK     ] accelerate      installed: 1.14.0       required: 1.14.0
[OK     ] torch           installed: 2.13.0       required: 2.13.0
[OK     ] numpy           installed: 2.4.6        required: 2.4.6
[OK     ] matplotlib      installed: 3.11.1       required: 3.11.1
[OK     ] pandas          installed: 3.0.5        required: 3.0.5
[OK     ] scikit-learn    installed: 1.9.0        required: 1.9.0

--- Hugging Face Connectivity ---
Status: ok
  Checkpoint: distilbert-base-uncased
  Vocab size: 30522

Environment report saved to: /Users/vientu/Deep Learning/TOTAL_LABPARACTICE_FOR_DEEP_LEARNING/total_practice/practice_3/docs/result/2026-08-10_phase01-envir

In [3]:
# Phase 2: Pretrained Sentiment Inference
from processing_own_phase.phase_02_pretrained_inference import (
    get_sentiment_pipeline,
    run_inference,
    inspect_tokenizer
)

# Load pipeline (first time downloads ~260MB model, requires internet)
pipe = get_sentiment_pipeline()

# 3 sample sentences
test_sentences = [
    "I absolutely loved this movie! The performances were outstanding.",
    "This film was a complete waste of time. Terrible acting.",
    "It was okay, nothing special."
]

# Run inference
results = run_inference(pipe, test_sentences)
for sentence, result in zip(test_sentences, results):
    print(f"Sentence: {sentence[:60]}...")
    print(f"  Label: {result['label']}, Score: {result['score']:.4f}\n")

# Inspect tokenizer
token_info = inspect_tokenizer(test_sentences[0])
print("Tokenizer vocab size:", token_info['vocab_size'])
print("Tokens (first 20):", token_info['tokens'][:20])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Sentence: I absolutely loved this movie! The performances were outstan...
  Label: POSITIVE, Score: 0.9999

Sentence: This film was a complete waste of time. Terrible acting....
  Label: NEGATIVE, Score: 0.9998

Sentence: It was okay, nothing special....
  Label: NEGATIVE, Score: 0.9821



Tokenizer vocab size: 30522
Tokens (first 20): ['[CLS]', 'i', 'absolutely', 'loved', 'this', 'movie', '!', 'the', 'performances', 'were', 'outstanding', '.', '[SEP]']


In [4]:
# Phase 3: Tokenization Investigation
from processing_own_phase.phase_03_tokenization import (
    tokenize_sentence_to_table,
    print_tokenized_table,
    decode_sanity_check,
    compare_tokenizers,
)

sentence = "I absolutely loved this movie! The performances were outstanding."
print(f"Raw sentence: {sentence}\n")

# Display the implementation's readable token table (tokens + token IDs).
token_table = tokenize_sentence_to_table(sentence)
print_tokenized_table(token_table)

# Display the attention mask produced by the same tokenizer comparison.
comparison = compare_tokenizers(sentence)
print(f"\nToken IDs: {comparison['finetuned']['input_ids']}")
print(f"Attention mask: {comparison['finetuned']['attention_mask']}")

# Verify that decoding preserves the sentence meaning.
decode_result = decode_sanity_check(sentence)
print(f"\nDecoded text: {decode_result['decoded_sentence']}")
print(f"Word overlap ratio: {decode_result['word_overlap_ratio']}")
print(f"Decode sanity check: {'PASS' if decode_result['semantically_consistent'] else 'FAIL'}")
assert decode_result['semantically_consistent'], "Decode sanity check failed"

result = comparison

# Print conclusion
print(f"Are tokenizers identical? {result['are_identical']}")
print(f"\nVocab size (fine-tuned): {result['finetuned']['vocab_size']}")
print(f"Vocab size (base): {result['base']['vocab_size']}")

if result['are_identical']:
    print("\nCONCLUSION: The tokenizers from the fine-tuned model and the base model are IDENTICAL.")
    print("  -> Fine-tuning only updates the backbone model weights.")
    print("  -> The tokenizer remains unchanged, vocab_size = 30522.")
    print("  -> This confirms that the tokenizer does not change during fine-tuning.")
else:
    print("\nCONCLUSION: Tokenizers are different (unexpected).")

Raw sentence: I absolutely loved this movie! The performances were outstanding.



Position  Token               Token ID    Special?  
----------------------------------------------------
0         [CLS]               101         Yes       
1         i                   1045        No        
2         absolutely          7078        No        
3         loved               3866        No        
4         this                2023        No        
5         movie               3185        No        
6         !                   999         No        
7         the                 1996        No        
8         performances        4616        No        
9         were                2020        No        
10        outstanding         5151        No        
11        .                   1012        No        
12        [SEP]               102         Yes       



Token IDs: [101, 1045, 7078, 3866, 2023, 3185, 999, 1996, 4616, 2020, 5151, 1012, 102]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]



Decoded text: i absolutely loved this movie! the performances were outstanding.
Word overlap ratio: 1.0
Decode sanity check: PASS
Are tokenizers identical? True

Vocab size (fine-tuned): 30522
Vocab size (base): 30522

CONCLUSION: The tokenizers from the fine-tuned model and the base model are IDENTICAL.
  -> Fine-tuning only updates the backbone model weights.
  -> The tokenizer remains unchanged, vocab_size = 30522.
  -> This confirms that the tokenizer does not change during fine-tuning.


In [5]:
# Phase 4: Dataset Loading
from processing_own_phase.phase_04_dataset_loading import (
    load_rotten_tomatoes,
    verify_dataset_contract,
    get_dataset_stats,
    check_null_values,
    print_sample_rows,
    save_dataset_summary,
)

# Load dataset
dataset = load_rotten_tomatoes()
print(f"Dataset loaded! Splits: {list(dataset.keys())}")

# Verify all required splits, sizes, schema and every label.
dataset_contract = verify_dataset_contract(dataset)
assert dataset_contract['all_pass'], "Phase 4 dataset contract failed"
print("\nPhase 4 verification:")
print(f"  Splits exist: {dataset_contract['splits_exist']}")
print(f"  Split sizes: {dataset_contract['split_sizes']}")
print(f"  Schema: {dataset_contract['schema']}")
print(f"  Label sets: {dataset_contract['label_sets']}")

# Get statistics
stats = get_dataset_stats(dataset)
print("\nDataset Statistics:")
for split_name, split_stats in stats.items():
    print(f"\n{split_name.capitalize()}:")
    print(f"  Total: {split_stats['total']}")
    print(f"  Positive: {split_stats['positive']} ({split_stats['positive_pct']:.1f}%)")
    print(f"  Negative: {split_stats['negative']} ({split_stats['negative_pct']:.1f}%)")

# Check null values
null_report = check_null_values(dataset)
print("\nNull Values Check:")
all_clean = True
for split_name, null_counts in null_report.items():
    for col, count in null_counts.items():
        if count > 0:
            print(f"  {split_name}.{col}: {count} null values")
            all_clean = False
        else:
            print(f"  {split_name}.{col}: No null values")

if all_clean:
    print("\nDataset is clean! No null values found.")

dataset_summary = {
    'contract': dataset_contract,
    'statistics': stats,
    'null_report': null_report,
}
dataset_summary_path = save_dataset_summary(dataset_summary)
print(f"Phase 4 verification: PASS")
print(f"Dataset summary saved: {dataset_summary_path}")

# Print sample rows
print_sample_rows(dataset, "train", 3)

Dataset loaded! Splits: ['train', 'validation', 'test']

Phase 4 verification:
  Splits exist: True
  Split sizes: {'train': 8530, 'validation': 1066, 'test': 1066}
  Schema: {'train': {'columns': ['label', 'text'], 'has_text': True, 'has_label': True}, 'validation': {'columns': ['label', 'text'], 'has_text': True, 'has_label': True}, 'test': {'columns': ['label', 'text'], 'has_text': True, 'has_label': True}}
  Label sets: {'train': [0, 1], 'validation': [0, 1], 'test': [0, 1]}

Dataset Statistics:

Train:
  Total: 8530
  Positive: 4265 (50.0%)
  Negative: 4265 (50.0%)

Validation:
  Total: 1066
  Positive: 533 (50.0%)
  Negative: 533 (50.0%)

Test:
  Total: 1066
  Positive: 533 (50.0%)
  Negative: 533 (50.0%)

Null Values Check:
  train.text: No null values
  train.label: No null values
  validation.text: No null values
  validation.label: No null values
  test.text: No null values
  test.label: No null values

Dataset is clean! No null values found.


Phase 4 verification: PASS
Dataset summary saved: /Users/vientu/Deep Learning/TOTAL_LABPARACTICE_FOR_DEEP_LEARNING/total_practice/practice_3/docs/result/phase_04_dataset_summary.json

--- First 3 rows of 'train' split ---

Row 1:
  Text: the rock is destined to be the 21st century's new " conan " and that he's going to make a splash eve...
  Label: 1 (POSITIVE)

Row 2:
  Text: the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column ...
  Label: 1 (POSITIVE)

Row 3:
  Text: effective but too-tepid biopic
  Label: 1 (POSITIVE)


In [6]:
# Phase 5: EDA and Sanity Checks
from processing_own_phase.phase_05_dataset_eda import (
    build_eda_summary,
    plot_token_length_distribution,
    save_eda_summary,
    save_token_length_statistics,
)
from processing_own_phase.phase_06_preprocessing import get_tokenizer
from processing_own_phase.config import MAX_TOKEN_LENGTH

# Load tokenizer base for token length computation
tokenizer = get_tokenizer()

# Run complete EDA through the Phase 5 implementation.
eda_summary = build_eda_summary(dataset, tokenizer)
assert eda_summary['all_pass'], "Phase 5 EDA sanity checks failed"
stats = eda_summary['train_length_statistics']
print("Text Length Statistics (Train):")
for key, value in stats.items():
    print(f"  {key}:")
    for k, v in value.items():
        print(f"    {k}: {v:.2f}")

recommendation = eda_summary['token_length_recommendation']
print("\nToken-length evidence:")
for key, value in recommendation.items():
    print(f"  {key}: {value:.2f}")

# Plot and save histogram
plot_token_length_distribution(
    dataset, tokenizer, "train",
    max_length_reference=MAX_TOKEN_LENGTH
)

print(f"\nSchema: {eda_summary['schema']}")
print(f"Text quality: {eda_summary['text_quality']}")
print(f"Within-split duplicates: {eda_summary['within_split_duplicates']}")
print(f"Cross-split overlap: {eda_summary['cross_split_overlap']}")
balance = eda_summary['label_distribution']
print("Label distribution:")
for split, data in balance.items():
    print(f"  {split}: Positive={data['positive']} ({data['positive_pct']:.1f}%), Negative={data['negative']} ({data['negative_pct']:.1f}%)")

# Final max_length decision
assert MAX_TOKEN_LENGTH >= recommendation['max']
eda_summary['selected_max_length'] = MAX_TOKEN_LENGTH
eda_summary['max_length_justification'] = (
    f"P95={recommendation['p95']:.0f}, P99={recommendation['p99']:.0f}, "
    f"observed max={recommendation['max']:.0f}; selected {MAX_TOKEN_LENGTH} "
    "is the smallest convenient rounded value above the observed maximum."
)
eda_summary_path = save_eda_summary(eda_summary)
token_stats_path = save_token_length_statistics({
    'train_length_statistics': stats,
    'token_length_recommendation': recommendation,
    'selected_max_length': MAX_TOKEN_LENGTH,
    'justification': eda_summary['max_length_justification'],
})
print(f"\nFinal MAX_TOKEN_LENGTH: {MAX_TOKEN_LENGTH}")
print(eda_summary['max_length_justification'])
print(f"EDA summary saved: {eda_summary_path}")
print(f"Token-length statistics saved: {token_stats_path}")
print("Phase 5 EDA verification: PASS")

Text Length Statistics (Train):
  character:
    min: 4.00
    max: 267.00
    mean: 113.97
    median: 111.00
    p95: 204.00
    p99: 241.00
  word:
    min: 1.00
    max: 59.00
    mean: 20.99
    median: 20.00
    p95: 37.00
    p99: 45.00
  token:
    min: 3.00
    max: 78.00
    mean: 27.37
    median: 27.00
    p95: 47.00
    p99: 56.00

Token-length evidence:
  p95: 47.00
  p99: 56.00
  max: 78.00


Histogram saved to: /Users/vientu/Deep Learning/TOTAL_LABPARACTICE_FOR_DEEP_LEARNING/total_practice/practice_3/docs/result/phase_05_token_length_distribution.png

Schema: {'train': True, 'validation': True, 'test': True}
Text quality: {'train': {'null_text': 0, 'empty_string': 0, 'whitespace_only': 0}, 'validation': {'null_text': 0, 'empty_string': 0, 'whitespace_only': 0}, 'test': {'null_text': 0, 'empty_string': 0, 'whitespace_only': 0}}
Within-split duplicates: {'train': 0, 'validation': 0, 'test': 0}
Cross-split overlap: {'train<->validation': 0, 'train<->test': 0, 'validation<->test': 0}
Label distribution:
  train: Positive=4265 (50.0%), Negative=4265 (50.0%)
  validation: Positive=533 (50.0%), Negative=533 (50.0%)
  test: Positive=533 (50.0%), Negative=533 (50.0%)

Final MAX_TOKEN_LENGTH: 80
P95=47, P99=56, observed max=78; selected 80 is the smallest convenient rounded value above the observed maximum.
EDA summary saved: /Users/vientu/Deep Learning/TOTAL_LABPARACTICE_FOR_DEEP_L

In [7]:
# Phase 6: Tokenizer and Preprocessing
from processing_own_phase.phase_06_preprocessing import (
    get_tokenizer,
    tokenize_dataset,
    show_tokenized_sample,
    check_tokenized_sample,
    decode_sanity_check,
    verify_preprocessing,
    verify_dynamic_padding,
    save_preprocessing_summary,
)

# Load tokenizer (base)
tokenizer = get_tokenizer()
print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"Vocab size: {tokenizer.vocab_size}")

# Tokenize dataset
tokenized_dataset = tokenize_dataset(dataset, tokenizer)
print(f"Tokenized dataset splits: {list(tokenized_dataset.keys())}")

# Show a sample
show_tokenized_sample(tokenized_dataset, "train", 0)

# Call sample and decode sanity checks explicitly.
sample_check = check_tokenized_sample(tokenized_dataset, "train", 0)
decode_checks = decode_sanity_check(tokenized_dataset, tokenizer, dataset, "train", 3)
assert sample_check['all_pass']
assert all(check['pass'] for check in decode_checks)
print(f"\nTokenized sample check: {sample_check}")
print(f"Decode sanity checks: {decode_checks}")

# Verify all three splits and dynamic padding with unequal-length samples.
preprocessing_summary = verify_preprocessing(dataset, tokenized_dataset, tokenizer)
dynamic_padding = verify_dynamic_padding(tokenized_dataset, tokenizer)
assert preprocessing_summary['all_pass']
assert dynamic_padding['all_pass']
preprocessing_summary['dynamic_padding'] = dynamic_padding
preprocessing_summary_path = save_preprocessing_summary(preprocessing_summary)
print(f"\nSplit verification: {preprocessing_summary['splits']}")
print(f"Dynamic-padding verification: {dynamic_padding}")
print(f"Preprocessing summary saved: {preprocessing_summary_path}")
print("Phase 6 preprocessing verification: PASS")

Tokenizer loaded: BertTokenizer
Vocab size: 30522
Tokenized dataset splits: ['train', 'validation', 'test']
Sample from 'train' split (index 0):
  input_ids length: 47
  input_ids (first 20): [101, 1996, 2600, 2003, 16036, 2000, 2022, 1996, 7398, 2301, 1005, 1055, 2047, 1000, 16608, 1000, 1998, 2008, 2002, 1005]
  attention_mask length: 47
  attention_mask (first 20): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  labels: 1
  Contains padding? False

Tokenized sample check: {'has_input_ids': True, 'has_attention_mask': True, 'has_labels': True, 'label_valid': True, 'lengths_match': True, 'all_pass': True}
Decode sanity checks: [{'index': 0, 'original_text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'decoded_text': 'the rock is destined to be the 21st century \' s new " conan " and that he \' s going to make a splash even greater th


Split verification: {'train': {'original_count': 8530, 'processed_count': 8530, 'sample_count_preserved': True, 'has_input_ids': True, 'has_attention_mask': True, 'has_labels': True, 'label_set': [0, 1], 'labels_valid': True, 'max_observed_sequence_length': 78, 'within_max_length': True, 'all_id_mask_lengths_match': True, 'all_pass': True}, 'validation': {'original_count': 1066, 'processed_count': 1066, 'sample_count_preserved': True, 'has_input_ids': True, 'has_attention_mask': True, 'has_labels': True, 'label_set': [0, 1], 'labels_valid': True, 'max_observed_sequence_length': 72, 'within_max_length': True, 'all_id_mask_lengths_match': True, 'all_pass': True}, 'test': {'original_count': 1066, 'processed_count': 1066, 'sample_count_preserved': True, 'has_input_ids': True, 'has_attention_mask': True, 'has_labels': True, 'label_set': [0, 1], 'labels_valid': True, 'max_observed_sequence_length': 67, 'within_max_length': True, 'all_id_mask_lengths_match': True, 'all_pass': True}}
Dynamic-